<a href="https://colab.research.google.com/github/comsadavid/AI-solutions/blob/main/competitions/mlcompete/Bank_Marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import LinearSVC

In [3]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [5]:
df_train.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,6456,33,admin.,married,secondary,no,2,yes,no,unknown,20,may,201,1,-1,0,unknown,0
1,5432,35,technician,married,secondary,no,545,yes,no,cellular,16,apr,1579,1,234,4,failure,1
2,6372,39,management,married,tertiary,no,763,yes,no,cellular,11,may,877,7,300,2,other,1
3,2254,37,management,married,tertiary,no,5106,no,no,cellular,30,apr,244,1,-1,0,unknown,1
4,8350,30,unemployed,married,tertiary,no,330,no,no,cellular,14,apr,111,1,-1,0,unknown,1


In [6]:
df_train.dtypes

id           int64
age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
deposit      int64
dtype: object

In [30]:
df_train.isna().sum().sum()

np.int64(0)

In [15]:
ocupatia = (df_train.groupby('job')['deposit'].mean()).idxmax()
output1 = pd.DataFrame({'subtaskID': [1], 'datapointID': [1], 'answer': ocupatia})
ocupatia

'student'

In [22]:
luna_max = df_train['month'].value_counts().idxmax()
output2 = pd.DataFrame({'subtaskID': [1], 'datapointID': [1], 'answer': luna_max})
luna_max

'may'

In [35]:
def encode_columns(df_train, df_test, columns):
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=int)
    encoded_train_list = ohe.fit_transform(df_train[columns])
    encoded_test_list = ohe.transform(df_test[columns])

    encoded_df_train = pd.DataFrame(encoded_train_list, columns=ohe.get_feature_names_out())
    encoded_df_test = pd.DataFrame(encoded_test_list, columns=ohe.get_feature_names_out())

    df_train[encoded_df_train.columns] = encoded_df_train
    df_test[encoded_df_test.columns] = encoded_df_test

    df_train.drop(columns, axis=1, inplace=True)
    df_test.drop(columns, axis=1, inplace=True)

def process(df_train, df_test):
    encode_list = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
    encode_columns(df_train, df_test, encode_list)
process(df_train, df_test)

C:\Users\david\AppData\Local\Temp\ipykernel_44404\2620108094.py:12: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_train.drop(columns, axis=1, inplace=True)
C:\Users\david\AppData\Local\Temp\ipykernel_44404\2620108094.py:13: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_test.drop(columns, axis=1, inplace=True)


In [36]:
df_train.head()

,id,age,balance,day,duration,campaign,pdays,previous,deposit,job_admin.,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
0,6456,33,2,20,201,1,-1,0,0,1,...,0,0,1,0,0,0,0,0,0,1
1,5432,35,545,16,1579,1,234,4,1,0,...,0,0,0,0,0,0,1,0,0,0
2,6372,39,763,11,877,7,300,2,1,0,...,0,0,1,0,0,0,0,1,0,0
3,2254,37,5106,30,244,1,-1,0,1,0,...,0,0,0,0,0,0,0,0,0,1
4,8350,30,330,14,111,1,-1,0,1,0,...,0,0,0,0,0,0,0,0,0,1


In [37]:
X = df_train.drop(['id', 'deposit'], axis=1)
y = df_train['deposit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
def evaluate_model(model):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print(f"Train: {f1_score(y_train, y_train_pred, average='macro')}, Test: {f1_score(y_test, y_test_pred, average='macro')}")

In [41]:
cbc = CatBoostClassifier()
cbc.fit(X_train, y_train)
evaluate_model(cbc)

Learning rate set to 0.023853
0:	learn: 0.6784878	total: 2.78ms	remaining: 2.77s
1:	learn: 0.6654414	total: 5.63ms	remaining: 2.81s
2:	learn: 0.6528672	total: 8.14ms	remaining: 2.71s
3:	learn: 0.6414828	total: 11ms	remaining: 2.73s
4:	learn: 0.6307114	total: 13.7ms	remaining: 2.72s
5:	learn: 0.6207241	total: 16.3ms	remaining: 2.69s
6:	learn: 0.6099495	total: 18.7ms	remaining: 2.65s
7:	learn: 0.5998736	total: 20.8ms	remaining: 2.58s
8:	learn: 0.5914594	total: 22.9ms	remaining: 2.52s
9:	learn: 0.5831836	total: 25ms	remaining: 2.47s
10:	learn: 0.5748743	total: 27.1ms	remaining: 2.44s
11:	learn: 0.5672494	total: 29.1ms	remaining: 2.39s
12:	learn: 0.5604520	total: 31.1ms	remaining: 2.36s
13:	learn: 0.5534803	total: 33.4ms	remaining: 2.35s
14:	learn: 0.5469728	total: 35.7ms	remaining: 2.35s
15:	learn: 0.5408253	total: 37.6ms	remaining: 2.31s
16:	learn: 0.5346706	total: 39.9ms	remaining: 2.31s
17:	learn: 0.5292562	total: 42ms	remaining: 2.29s
18:	learn: 0.5231953	total: 44.1ms	remaining: 2.28

In [48]:
lr = LinearSVC()
lr.fit(X_train, y_train)
evaluate_model(lr)

Train: 0.8304115182756081, Test: 0.8193657411443926


In [49]:
lr = XGBClassifier()
lr.fit(X_train, y_train)
evaluate_model(lr)

Train: 0.9663557529401624, Test: 0.8534487393150907


In [50]:
lr = LGBMClassifier()
lr.fit(X_train, y_train)
evaluate_model(lr)

[LightGBM] [Info] Number of positive: 3411, number of negative: 3732
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001163 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1005
[LightGBM] [Info] Number of data points in the train set: 7143, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.477530 -> initscore=-0.089939
[LightGBM] [Info] Start training from score -0.089939
Train: 0.9197604430261934, Test: 0.8585433680653336


In [ ]:
def solve(model):
    
